### Step 1: Load your saved model

Run this once your `.keras` model file exists.

In [1]:
from keras.models import load_model

model = load_model('equation_reader.keras')
categories = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '%', '[', ']']


I0000 00:00:1788617720.876294    2862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788617721.116808    2862 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788617722.405848    2862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788617724.446998    2862 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3700 MB mem

### Step 2: Preprocess the full scribble image

**Important, verified against the real dataset:** this model's training data (`michelheusser/handwritten-digits-and-operators`) uses **dark ink strokes on a light background** — like normal pen on paper — *not* the MNIST-style white-on-black convention. Checking a real sample from the dataset confirms strokes sit near pixel value 0 and background near 255.

So regardless of how your source photo looks, the output of this function must always land on **dark stroke (~0) / light background (~255)**. Use `symbols_are_dark_in_source` to tell it which case you're in:
- Dark ink on light paper (already matches) → `True`
- Light/bright strokes on a dark canvas or app background (needs flipping) → `False`

In [2]:
import cv2
import numpy as np

def preprocess_image(image_path, symbols_are_dark_in_source=True):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image at {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Output ALWAYS ends up dark-stroke/light-background, matching training data.
    thresh_type = cv2.THRESH_BINARY if symbols_are_dark_in_source else cv2.THRESH_BINARY_INV
    _, binary = cv2.threshold(blurred, 0, 255, thresh_type + cv2.THRESH_OTSU)

    return binary


### Step 2b: Thin the strokes to match the training data's style

The dataset's strokes were generated by an algorithm specifically built to keep them under ~1px wide at 28x28 — near-skeleton lines, not filled shapes. A real pen/finger/marker scribble is much thicker, and that gap alone can hurt accuracy even with polarity fixed. `cv2.ximgproc.thinning` (needs `opencv-contrib-python`) shrinks strokes toward a 1px skeleton without changing their shape.

In [3]:
def thin_strokes(binary_image):
    """binary_image: dark stroke (0) / light background (255) -- the output of preprocess_image."""
    inv = cv2.bitwise_not(binary_image)          # thinning expects white=foreground
    thinned_inv = cv2.ximgproc.thinning(inv)
    return cv2.bitwise_not(thinned_inv)          # flip back to dark-stroke/light-bg


### Step 3: Segment the image into individual symbols

Same merging logic as before, but note `cv2.findContours` expects a white foreground, so we invert before finding contours (the image itself stays dark-stroke/light-bg everywhere else).

In [4]:
def _get_raw_boxes(binary_image, min_area=40, min_dim=5):
    inv = cv2.bitwise_not(binary_image)  # contours need white=foreground
    contours, _ = cv2.findContours(inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area or w < min_dim or h < min_dim:
            continue
        boxes.append((x, y, w, h))
    return boxes

def segment_symbols(binary_image, min_area=40, min_dim=5, x_gap_thresh=15):
    """Returns a list of dicts: {'box': (x, y, w, h), 'strokes': n}, sorted left-to-right."""
    boxes = sorted(_get_raw_boxes(binary_image, min_area, min_dim), key=lambda b: b[0])
    merged, used = [], [False] * len(boxes)

    for i, (x1, y1, w1, h1) in enumerate(boxes):
        if used[i]:
            continue
        cx1, cy1, cx2, cy2 = x1, y1, x1 + w1, y1 + h1
        used[i] = True
        stroke_count = 1
        changed = True
        while changed:
            changed = False
            for j, (x2, y2, w2, h2) in enumerate(boxes):
                if used[j]:
                    continue
                bx1, by1, bx2, by2 = x2, y2, x2 + w2, y2 + h2
                h_close = not (bx1 > cx2 + x_gap_thresh or bx2 < cx1 - x_gap_thresh)
                if h_close:
                    used[j] = True
                    cx1, cy1 = min(cx1, bx1), min(cy1, by1)
                    cx2, cy2 = max(cx2, bx2), max(cy2, by2)
                    stroke_count += 1
                    changed = True
        merged.append({"box": (cx1, cy1, cx2 - cx1, cy2 - cy1), "strokes": stroke_count})

    return sorted(merged, key=lambda m: m["box"][0])


### Step 3b: Detect `=` as an end-of-equation marker

Unchanged from before — `=` is drawn as 2+ separate strokes forming a landscape-shaped box, which no other symbol in this category set does.

In [5]:
def is_equals_sign(entry, w_over_h_thresh=0.9):
    x, y, w, h = entry["box"]
    return entry["strokes"] >= 2 and (w / h) > w_over_h_thresh


A visual debug helper — red boxes are skipped as `=`, green boxes get classified.

In [6]:
import matplotlib.pyplot as plt

def visualize_segmentation(image_path, entries):
    img = cv2.imread(image_path)
    for entry in entries:
        x, y, w, h = entry["box"]
        color = (0, 0, 255) if is_equals_sign(entry) else (0, 255, 0)  # BGR
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


### Step 4: Normalize each cropped symbol to match training format

**Fixed:** the padding around each crop must use the *background* color. Since background is now light (255), pad with white, not black — padding with black here would put a dark border around every symbol, which the model has never seen and would hurt classification just as much as the polarity bug did.

In [7]:
def normalize_symbol(binary_image, box, target_size=28, padding_ratio=0.2):
    x, y, w, h = box
    crop = binary_image[y:y+h, x:x+w]

    side = max(w, h)
    pad = int(side * padding_ratio)
    side += pad * 2

    square = np.full((side, side), 255, dtype=np.uint8)  # pad with light background, not black
    y_off = (side - h) // 2
    x_off = (side - w) // 2
    square[y_off:y_off+h, x_off:x_off+w] = crop

    resized = cv2.resize(square, (target_size, target_size), interpolation=cv2.INTER_AREA)
    return resized.astype('float32') / 255.0


### Step 5: Classify each symbol and reconstruct the equation string

Unchanged logic — skips `=` boxes, classifies everything else.

In [8]:
def classify_symbols(model, binary_image, entries, categories):
    equation_chars = []
    for entry in entries:
        if is_equals_sign(entry):
            continue
        symbol = normalize_symbol(binary_image, entry["box"])
        symbol_input = np.expand_dims(symbol, axis=(0, -1))
        prediction = model.predict(symbol_input, verbose=0)
        class_index = int(np.argmax(prediction))
        equation_chars.append(categories[class_index])
    return ''.join(equation_chars)


### Step 6: Parse and solve the equation safely

Unchanged — note this already raises a clear error on invalid strings like `"88*"` rather than silently returning `0.0`. If you're seeing silent `0.0` results elsewhere, check whether your local copy of this function has a bare `except: return 0` somewhere — that hides exactly the kind of misclassification you're debugging.

In [9]:
import sympy

def solve_equation(equation_str, percent_as_modulo=True):
    expr_str = equation_str.replace('[', '(').replace(']', ')')

    if not percent_as_modulo:
        expr_str = expr_str.replace('%', '/100')

    try:
        expr = sympy.sympify(expr_str)
        return sympy.N(expr)
    except (sympy.SympifyError, TypeError, ValueError) as e:
        return f"Could not evaluate '{equation_str}' (parsed as '{expr_str}'): {e}"


### Step 7: Chain it into one end-to-end function

In [10]:
def scribble_to_answer(image_path, model, categories, symbols_are_dark_in_source=True,
                        apply_thinning=True, percent_as_modulo=True):
    binary = preprocess_image(image_path, symbols_are_dark_in_source=symbols_are_dark_in_source)
    if apply_thinning:
        binary = thin_strokes(binary)
    entries = segment_symbols(binary)
    if not entries:
        raise ValueError("No symbols detected — check preprocessing/threshold settings.")

    equation_str = classify_symbols(model, binary, entries, categories)
    result = solve_equation(equation_str, percent_as_modulo=percent_as_modulo)
    return equation_str, result, entries


### Step 8: Sanity-check the pipeline

Set `symbols_are_dark_in_source` based on your actual photo (see Step 2's note), then run this against your real scribbles and check the printed equation against what you actually wrote.

In [12]:
image_path = '/home/arc/Code/eq_idn/scib5.jpeg'  # replace with your actual scribble image

equation_str, result, entries = scribble_to_answer(
    image_path, model, categories, symbols_are_dark_in_source=False,apply_thinning=False
)
print("Detected symbols:", len(entries))
print("Recognized equation:", equation_str)
print("Result:", result)

visualize_segmentation(image_path, entries)


AttributeError: module 'cv2' has no attribute 'imread'

### Step 9: If accuracy is still off after this fix

The polarity fix should be the big jump. If classification is still shaky after that, the remaining gap is almost certainly stroke thickness/style. Two ways to close it further:

1. **At inference** (already added above): `thin_strokes()` shrinks your bold strokes toward the training data's thin style.
2. **At training time** (more robust, but requires retraining): augment your training set with a few *dilated* copies of each symbol (thicker versions) using `cv2.dilate`, so the model learns to recognize a range of thicknesses rather than only near-1px lines. This is generally more reliable than trying to perfectly replicate the dataset's original thinning algorithm at inference time.

In [ ]:
# Example of generating a thicker-stroke augmented copy for training (run in model_train.ipynb, not here):
# kernel = np.ones((2, 2), np.uint8)
# thicker = cv2.dilate(original_dark_on_light_image, kernel, iterations=1)
